### From Silver To Gold: Aggregation and KPI Tables

In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
dbutils.widgets.text("storage_account_name", "ecommcistorage05", "Storage Account Name")
dbutils.widgets.text("container_name", "ecomm-raw-data", "Container Name")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

In [0]:
df_batch = spark.read.table(f"{catalog_name}.silver.slv_shipments")
display(df_batch.limit(20))

order_dt,shipment_id,order_id,carrier,_rescued_data,ingest_timestamp,source_file,processed_time
2025-08-01,S643611,643611,BLUEDART,null,2026-09-27T13:37:53.997Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv,2026-09-28T11:21:51.753Z
2025-08-01,S643612,643612,AUSPOST,null,2026-09-27T13:37:53.997Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv,2026-09-28T11:21:51.753Z
2025-08-01,S643613,643613,DHL,null,2026-09-27T13:37:53.997Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv,2026-09-28T11:21:51.753Z
2025-08-01,S643614,643614,XPRESSBEES,null,2026-09-27T13:37:53.997Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv,2026-09-28T11:21:51.753Z
2025-08-01,S643615,643615,DELHIVERY,null,2026-09-27T13:37:53.997Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv,2026-09-28T11:21:51.753Z
2025-08-01,S643616,643616,BLUEDART,null,2026-09-27T13:37:53.997Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv,2026-09-28T11:21:51.753Z
2025-08-01,S643617,643617,DHL,null,2026-09-27T13:37:53.997Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv,2026-09-28T11:21:51.753Z
2025-08-01,S643618,643618,DHL,null,2026-09-27T13:37:53.997Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv,2026-09-28T11:21:51.753Z
2025-08-01,S643619,643619,DHL,null,2026-09-27T13:37:53.997Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv,2026-09-28T11:21:51.753Z
2025-08-01,S643620,643620,DHL,null,2026-09-27T13:37:53.997Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv,2026-09-28T11:21:51.753Z


In [0]:
spark.sql(f"USE CATALOG {catalog_name}")

DataFrame[]

In [0]:
# readChangeFeed flag is used to read the change feed (_change_type column mainly)
df = spark.readStream \
.format("delta") \
.option("readChangeFeed", "true") \
.table(f"{catalog_name}.silver.slv_shipments")

In [0]:
df_union = df.filter("_change_type IN ('insert', 'update_postimage')")

### Add `carrier_group`:
### Domestic: ECOMEXPRESS, DELHIVERY, XPRESSBEES, BLUEDART.
### International: All other carriers.

In [0]:
df_union = df_union.withColumn("carrier_group",F.when(F.col("carrier").isin("ECOMEXPRESS","DELHIVERY","XPRESSBEES","BLUEDART"),"Domestic").otherwise("International"))

### Add `is_weekend_shipment` flag:
### True if `order_dt` is Saturday or Sunday, else False.

In [0]:
df_union = df_union.withColumn("is_weekend_shipment",F.when(F.dayofweek(F.col("order_dt")).isin(1,7),True).otherwise(False))

In [0]:
df = df_union.select('order_dt','shipment_id', 'order_id', 'carrier', 'carrier_group', 'is_weekend_shipment', '_rescued_data', 'ingest_timestamp', 'source_file')

### Gold checkpoint

In [0]:
gold_check_point = f'abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/gold/fact_shipments/'

In [0]:
def upsert_to_gold(microBatchDF, batchId):
    table_name = f"{catalog_name}.gold.gld_fact_shipments"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("gold_table").merge(
            microBatchDF.alias("batch_table"),
            "gold_table.order_id = batch_table.order_id" and "gold_table.shipment_id = batch_table.shipment_id"
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
df.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_gold
).format("delta").option("checkpointLocation", gold_check_point).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).trigger(
    once=True
).start().awaitTermination()